# vgscp — full REAL multi-seed run (E1–E4) on Colab GPU

Runs the four experiments end-to-end on **real** Waterbirds + CUB-200 data with frozen CLIP ViT-B/32 features (encoded once, cached) and logistic heads — **no large-model training**. Run top-to-bottom on a **GPU** runtime (`Runtime → Change runtime type → GPU`).

Honesty: the pre-committed verdicts (`eval/e1_verdict.py`, `eval/scacp_gate.py`) are LOCKED. Whatever the real multi-seed numbers are — including FALLBACK / ties / softenings — is what gets reported.

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE    = "git"          # "git" or "drive"
REPO_URL       = "https://github.com/<YOUR_USER>/vgscp.git"   # EDIT (private: https://<TOKEN>@github.com/<user>/vgscp.git)
REPO_BRANCH    = "main"
REPO_DRIVE_ZIP = "/content/drive/MyDrive/vgscp.zip"           # used only if REPO_SOURCE=="drive"

DRIVE_CACHE    = "/content/drive/MyDrive/vgscp_cache"  # datasets + CLIP feature cache persisted here
SEEDS          = 10                                    # >=10 random cal/test splits per spec

# Dataset URLs — EDIT IF URL CHANGES
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CUB_URL        = "https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz"
# ======================================================
import os, time, subprocess, sys
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)
def run_module(mod_args, label):
    t = time.time(); print(f"\n===== {label} =====")
    p = subprocess.run([sys.executable, "-m", *mod_args])
    dt = time.time() - t; print(f"[{label}] exit={p.returncode}  wall={dt/60:.1f} min")
    if dt > 4.5 * 3600: print(f"[WARN] {label} approaching the 5h cap")
    return p.returncode

## 1. GPU check + install

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("[WARN] No GPU — CLIP encode will be slow. Set Runtime->GPU.")
import subprocess
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn pandas matplotlib", shell=True)

## 2. Mount Drive (persist datasets + CLIP feature cache across restarts)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
print("cache dir:", DRIVE_CACHE)

## 3. Get the repo

In [ ]:
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
    subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(f"{REPO_DIR}/{d}")]
    if len(subs) == 1 and not os.path.exists(f"{REPO_DIR}/scripts"):
        inner = f"{REPO_DIR}/{subs[0]}"; sh(f"shopt -s dotglob && mv {inner}/* {REPO_DIR}/ && rmdir {inner}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 4. Datasets — download (cached to Drive) + extract, set env vars

In [ ]:
def fetch(url, drive_name, extract_to):
    os.makedirs(extract_to, exist_ok=True)
    tarball = os.path.join(DRIVE_CACHE, drive_name)
    if not os.path.exists(tarball):
        sh(f"wget -q -O '{tarball}' '{url}'")
    else:
        print("cached tarball:", tarball)
    sh(f"tar -xzf '{tarball}' -C '{extract_to}'")
    return extract_to

fetch(WATERBIRDS_URL, "waterbirds.tar.gz", "/content/data/waterbirds")
fetch(CUB_URL, "CUB_200_2011.tgz", "/content/data/cub")
os.environ["WATERBIRDS_ROOT"] = "/content/data/waterbirds"
os.environ["CUB_ROOT"] = "/content/data/cub"
# point the repo CLIP feature cache at Drive so re-runs skip re-encoding
sh("rm -rf results/cache_clip"); os.makedirs("results", exist_ok=True)
os.makedirs(f"{DRIVE_CACHE}/clip", exist_ok=True)
sh(f"ln -s {DRIVE_CACHE}/clip results/cache_clip")
print("WATERBIRDS_ROOT=", os.environ["WATERBIRDS_ROOT"]); print("CUB_ROOT=", os.environ["CUB_ROOT"])

## 5. Encode + cache CLIP features ONCE (later cells reuse the cache)

In [ ]:
t = time.time()
from config_util import load_config
from experiments.real_data import load_real_bundle
cfg = load_config("configs/cub200_frontier.yaml")
bundle = load_real_bundle(cfg, seed=0)
print("feature shapes:", {k: v.shape for k, v in bundle.features.items()})
print("species present:", bundle.info["n_species_present"], "| CUB join:", bundle.info["cub_join"]["coverage"])
print(f"[encode+cache] wall={(time.time()-t)/60:.1f} min (cached to Drive)")

## 6. IN-DOMAIN unified 2×2 — diagnostic-first, gate-first (run spec v4)
The species head is now trained **in-domain on the composited `train` split** (v3 wrongly trained on
clean CUB-200 and applied to composited → 0.700 clean vs 0.246 composited, and a clean head never
learns the shortcut). The run prints the **§1 diagnostic** first — clean→clean (anchor),
clean→composited (the v3 mismatch), and the DECISIVE **in-domain** composited→composited d_test top-1
(all / typical / atypical) — then applies the **§2 decision table**. **§4 HARD HALT:** if the
**in-domain typical** top-1 is **< 0.55**, the orchestrator writes `BLOCKERS_v4.md`, emits **NO
2×2/verdict**, and the next cell **stops the notebook so E2/E4 never run** (branch C → recommend
coarser labels; do NOT patch). Only on a pass do the 2×2 + hardened verdict (≥2/3 scores incl. ρ=0.5)
run. **Report whatever it yields — including FALLBACK.**

In [ ]:
# §4 GATE-FIRST: the unified run halts (exit!=0) + writes BLOCKERS_v4.md if in-domain typical < 0.55.
rc = run_module(["scripts.run_unified_2x2", "--config", "configs/cub200_frontier.yaml",
                 "--seeds", str(SEEDS), "--concept-source", "cbm"], "UNIFIED-cbm")
if rc != 0:
    raise RuntimeError("§4 IN-DOMAIN GATE FAILED — see BLOCKERS_v4.md for the §1 diagnostic + branch "
                       "(B=encoding bug / C=construction destroys signal). No 2×2/verdict; E2/E4 "
                       "SKIPPED. For branch C, drop to coarser labels; do NOT patch or lower the gate.")
# appendix: CLIP zero-shot concept (leakage-free robustness check)
run_module(["scripts.run_unified_2x2", "--config", "configs/cub200_frontier.yaml", "--seeds", str(SEEDS),
            "--concept-source", "zeroshot", "--out", "results/unified_zeroshot"], "UNIFIED-zeroshot")

import pandas as pd, json
from IPython.display import Image, display
r = json.load(open("results/unified/unified_results.json"))
dg = r.get("diag") or {}
print(f"§1 diagnostic: clean->clean(anchor)={dg.get('clean_to_clean')}  "
      f"clean->composited(mismatch)={dg.get('clean_to_composited')}  "
      f"in-domain all/typ/atyp={dg.get('indomain_all')}/{dg.get('indomain_typical')}/{dg.get('indomain_atypical')}")
print(f"§4 gate: in-domain typical top-1={r['feat_top1_indomain_typical']:.3f} >= 0.55 (PASSED) | "
      f"clean-CUB anchor={r['feat_top1_cleancub']:.3f} | concept[{r['concept_source']}]={r['cpt_top1']:.3f}")
print("HEADLINE (>=2/3 scores):", r["combined"]["label"], "—", r["combined"]["rationale"])
for s in ("APS", "RAPS", "THR"):
    v = r["verdicts"][s]
    print(f"  {s}: {'GREEN' if v['green'] else 'FALLBACK'} (majority={v['majority']}, "
          f"rho=0.5 recovers={v['hardest_recovers']}, R={v['sweep_mean_R']:.2f})")
df = pd.read_csv("results/unified/unified_2x2.csv")
display(df[df.score == "APS"].groupby(["representation", "scheme", "test_corr"])
        .agg(worst_cov=("worst_cov", "mean"), set_size=("mean_set_size", "mean"),
             gap=("cov_gap", "mean"), marg=("marg_cov", "mean")).round(3))
for nm in ("u2x2_gap_vs_rho_APS", "u2x2_frontier_APS"):
    p = f"results/figures/{nm}.png"
    if os.path.exists(p): display(Image(p))

## 7. §1 diagnostic recap + §2e accuracy-controlled efficiency
The former **E3** is folded into the unified 2×2 above (the gap-vs-ρ figure *is* the E3 view). Below:
the §1 diagnostic accuracies, the **§4 in-domain gate** number (in-domain typical top-1), and the §2e
accuracy-matched efficiency (concept vs feature set size) — reported **only with the accuracy caveat**,
now matched at a *competent* accuracy rather than v3's ~0.15.

In [ ]:
import json
r = json.load(open("results/unified/unified_results.json"))
ac, eff, dg = r["acc_control"], r["efficiency"], (r.get("diag") or {})
print("§1 diagnostic accuracies:", {k: round(v, 3) for k, v in dg.items()})
print(f"§4 gate = in-domain TYPICAL top-1 = {r['feat_top1_indomain_typical']:.3f} (>=0.55 to proceed; "
      f"v3 ran on a 0.246 in-domain head measured by the wrong clean-CUB gate).")
print(f"   heads: feature(pool)={r['feat_top1']:.3f}  concept[{r['concept_source']}]={r['cpt_top1']:.3f}  "
      f"clean-CUB anchor={r['feat_top1_cleancub']:.3f}")
print(f"§2e accuracy-match feasible={ac['feasible']} (n_matched_classes={ac['n_matched_classes']}, tol={ac['tol']})")
if not ac["feasible"]:
    print("   -> efficiency CONFOUNDED BY ACCURACY (labelled as such; no representation-driven claim).")
import pandas as pd
display(pd.DataFrame(eff["per_rho"]).round(3))
v = r["verdicts"]["APS"]
print(f"Mechanism main effect (Mondrian gap reduction, sweep mean): feature={v['sweep_mean_mech_feat']:+.3f}"
      f"  concept={v['sweep_mean_mech_cpt']:+.3f}  |  recovered fraction R sweep-mean={v['sweep_mean_R']:.2f}")

## 8. E2 — verifiability collapse, RE-RUN with PREDICTED concepts (run spec v3 §3)
v2 E2 used the **leaky ground-truth** attributes (concept_trust 0.957, std 0.0). v3 re-runs E2 with
the **same image-derived `cbm` predicted concepts as the 2×2** (`--concept-source cbm`), so the
verifiability comparison rests on honest concepts. **Only runs if the §1.4 gate passed above.**
**Report whatever appears** — the honest predicted concept_trust was ~0.60 AUROC in v2, so
verifiability may now *beat* it and **flip the falsification**; state that plainly if so.

In [ ]:
run_module(["scripts.run_e2_verifiability_multiseed", "--config", "configs/premise2_waterbirds.yaml",
            "--seeds", str(SEEDS), "--concept-source", "cbm"], "E2-predicted")
import pandas as pd
d = pd.read_csv("results/e2/e2_verifiability_metrics.csv")
g = d.groupby(["space", "signal"]).agg(min_auroc=("minority_auroc", "mean"),
                                       contam=("contamination_auroc", "mean")).round(3)
display(g)
# did honest verifiability beat the honest predicted concept_trust? (the falsification may FLIP)
for space in ("attributes_only", "mixed"):
    try:
        ct = g.loc[(space, "concept_trust"), "min_auroc"]; vf = g.loc[(space, "V_full"), "min_auroc"]
        print(f"[{space}] concept_trust={ct:.3f}  V_full={vf:.3f}  -> "
              f"{'verifiability ADDS lift (falsification FLIPS)' if vf > ct else 'concept_trust still >= verifiability'}")
    except KeyError:
        pass

## 9. E4 — scacp 312-attribute locked gate  *(clean negative — NOT re-run per spec v2 §3)*
E4 was a clean negative; spec v2 says **reporting fixes only**. Corrected reporting (`RESULTS_v2.md
§5`): **1/311 pass** (not 0/312), **median differential-noise AUROC 0.596** (not ≈0.48) — a clean
negative; the single pass is expected at this multiplicity. The cell below only regenerates the scan.

In [ ]:
run_module(["scripts.run_e4_scacp_gate", "--real", "--config", "configs/cub200_frontier.yaml"], "E4")
import json, pandas as pd
r = json.load(open("results/e4/e4_results.json"))
print(f"GATE PASSES: {r['n_pass']}/{r['n_attributes']}  median diff-noise AUROC={r['median_diff_auroc']:.3f}")
print(f"per-criterion: diff>=0.70:{r['pass_diff']}  gap>=0.03:{r['pass_gap']}  support>=100:{r['pass_support']}")
d = pd.read_csv("results/e4/e4_scacp_gate_scan.csv")
display(d["diff_noise_auroc"].describe().round(3))

## 10. Consolidate + copy to Drive + zip for download

In [ ]:
import json, datetime, platform
summary = {"date": str(datetime.date.today()), "seeds": SEEDS, "platform": platform.platform()}
# v4: in-domain unified 2×2 (gate-gated) + in-domain predicted-concept E2; E4 not re-run
for tag, p in (("unified_cbm", "results/unified/unified_results.json"),
               ("unified_zeroshot", "results/unified_zeroshot/unified_results.json"),
               ("e2_predicted", "results/e2/e2_results.json"), ("e4", "results/e4/e4_results.json")):
    if os.path.exists(p): summary[tag] = json.load(open(p))
open("results/REAL_RUN_v4_SUMMARY.json", "w").write(json.dumps(summary, indent=2, default=str))
out = f"{DRIVE_CACHE}/results_real_v4"
sh(f"rm -rf {out} && cp -r results {out}")
for f in ("RESULTS_v4.md", "BLOCKERS_v4.md", "CHANGELOG_v4.md", "RESULTS_v3.md", "BLOCKERS_v3.md"):
    if os.path.exists(f): sh(f"cp {f} {out}/ 2>/dev/null || true")
sh(f"cd {DRIVE_CACHE} && zip -qr results_real_v4.zip results_real_v4")
print("Saved to Drive:", out, "and", f"{DRIVE_CACHE}/results_real_v4.zip")
try:
    from google.colab import files; files.download(f"{DRIVE_CACHE}/results_real_v4.zip")
except Exception as e: print("download skipped:", e)
print("\nNOTE: reached ONLY if the §4 in-domain gate passed. Report whatever the in-domain 2×2 + "
      "predicted-concept E2 produce — GREEN or FALLBACK. If the gate FAILED, the run halted at cell 6 "
      "with BLOCKERS_v4.md (branch B/C), no verdict, E2 skipped.")